In [1]:
import json
from datasets import Dataset

train_file_path = "../new_data/train_large.json"
val_file_path = "../new_data/val_large.json"
test_file_path = "../new_data/test_large.json"

# Chuyển dữ liệu thành định dạng phù hợp cho Hugging Face Dataset
data_processed = []
# Đọc file JSON
with open(train_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_processed.append({
            "input": data["text"],
            "output": data["label"]
        })

data_validate = []
# Đọc file JSON
with open(val_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_validate.append({
            "input": data["text"],
            "output": data["label"]
        })

data_test = []
# Đọc file JSON
with open(test_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_test.append({
            "input": data["text"],
            "output": data["label"]
        })
# Chuyển đổi dữ liệu thành Dataset của Hugging Face
train_data = Dataset.from_dict({
    'input': [item['input'] for item in data_processed],
    'output': [item['output'] for item in data_processed]
})

val_data = Dataset.from_dict({
    'input': [item['input'] for item in data_validate],
    'output': [item['output'] for item in data_validate]
})

test_data = Dataset.from_dict({
    'input': [item['input'] for item in data_test],
    'output': [item['output'] for item in data_test]
})


/home/creator/miniconda3/envs/toanpn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from torch.utils.data import Dataset as dt

def tokenize_data(data, tokenizer, max_length=128, label_map=None):

    # Auto-generate a label map if not provided
    if label_map is None:
        unique_labels = sorted({item["output"] for item in data})
        label_map = {label: idx for idx, label in enumerate(unique_labels)}

    # Tokenize the dataset
    tokenized_data = []
    for example in data:
        input_text = example["input"]
        output_label = example["output"]

        # Tokenize input text
        encoded_input = tokenizer(
            input_text,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        # Map output label to numeric ID
        label_id = label_map[output_label]

        # Append tokenized data
        tokenized_data.append({
            "input_ids": encoded_input["input_ids"].squeeze(),
            "attention_mask": encoded_input["attention_mask"].squeeze(),
            "labels": label_id
        })

    return tokenized_data, label_map


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [6]:

# Áp dụng hàm tiền xử lý cho dữ liệu
train_dataset, train_label_map = tokenize_data(train_data, tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_dataset, val_label_map = tokenize_data(val_data, tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=True)

test_dataset, test_label_map = tokenize_data(test_data, tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [7]:
print(train_dataset[0])
print(f"{len(train_dataset)}")
print(train_label_map)
print(val_dataset[0])
print(f"{len(val_dataset)}")
print(val_label_map)
print(test_dataset[0])
print(f"{len(test_dataset)}")
print(test_label_map)

{'input_ids': tensor([    0, 16265,     5, 54133, 63175,    87, 18822, 24145, 20271,    70,
         5155,   136, 24145,  4568, 17431,     5,     2,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1, 

In [8]:
from transformers import AdamW, get_linear_schedule_with_warmup, AutoModelForSequenceClassification, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm  # Dùng để hiển thị thanh tiến độ

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(train_label_map))  # Đảm bảo số lớp đúng

# Tạo optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Cấu hình thiết bị
device = torch.device("cuda")
model.to(device)

# Số epoch
epochs = 3

# Định nghĩa số bước warmup và tổng số bước
num_training_steps = len(train_dataloader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)

# Scheduler để điều chỉnh learning rate
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

# Huấn luyện mô hình
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{epochs}"):
        # Chuyển dữ liệu vào GPU (nếu có)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1} - Average Loss: {avg_loss:.4f}")


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/creator/miniconda3/envs/toanpn/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_

Epoch 1 - Average Loss: 0.0563


Epoch 2/3: 100%|██████████| 4254/4254 [1:25:33<00:00,  1.21s/it]


Epoch 2 - Average Loss: 0.0185


Epoch 3/3: 100%|██████████| 4254/4254 [1:25:34<00:00,  1.21s/it]

Epoch 3 - Average Loss: 0.0127


In [54]:
from transformers import AdamW, get_linear_schedule_with_warmup, T5ForConditionalGeneration, T5Tokenizer

import torch
from tqdm import tqdm  # Dùng để hiển thị thanh tiến độ trong huấn luyện


# Load pre-trained model and tokenizer
model_name = 't5-small'  # Or use 't5-base' or 't5-large' for more capacity
model = T5ForConditionalGeneration.from_pretrained("../my_t5_model")
tokenizer = T5Tokenizer.from_pretrained("../my_t5_model")
# In thông số của mô hình
print("Mô hình:", model_name)
print("Số tham số:", sum(p.numel() for p in model.parameters()))
print("Số lớp (layers):", model.config.num_hidden_layers)
print("Kích thước embedding:", model.config.hidden_size)
print("Số đầu attention:", model.config.num_attention_heads)
print("Kích thước vocab:", len(tokenizer))

Mô hình: t5-small
Số tham số: 60506624
Số lớp (layers): 6
Kích thước embedding: 512
Số đầu attention: 8
Kích thước vocab: 32100


In [48]:
# In thông số của mô hình
print("Mô hình:", model_name)
print("Số tham số:", sum(p.numel() for p in model.parameters()))
print("Số lớp (layers):", model.config.num_hidden_layers)
print("Kích thước embedding:", model.config.hidden_size)
print("Số đầu attention:", model.config.num_attention_heads)
print("Chiều dài tối đa của input:", model.config.max_position_embeddings)
print("Kích thước vocab:", len(tokenizer))

Mô hình: xlm-roberta-base
Số tham số: 278045955
Số lớp (layers): 12
Kích thước embedding: 768
Số đầu attention: 12
Chiều dài tối đa của input: 514
Kích thước vocab: 250002


In [9]:
# Lưu mô hình dưới dạng SavedModel
model.save_pretrained('../my_xrb_model')
tokenizer.save_pretrained('../my_xrb_model')  # Lưu luôn tokenizer để tái sử dụng

('../my_xrb_model/tokenizer_config.json',
 '../my_xrb_model/special_tokens_map.json',
 '../my_xrb_model/sentencepiece.bpe.model',
 '../my_xrb_model/added_tokens.json',
 '../my_xrb_model/tokenizer.json')

In [10]:
import torch
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup

# Khởi tạo danh sách để lưu nhãn thực tế và dự đoán
y_true = []
y_pred = []
loss_values = []

# Chuyển model sang chế độ đánh giá
model.eval()

# Thiết bị tính toán
device = torch.device("cuda")
model.to(device)

# Tính Loss và dự đoán
# Không tính toán gradient trong khi đánh giá
with torch.no_grad():
    for batch in tqdm(val_dataloader, desc="Validating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        # Tính Loss
        loss = outputs.loss
        loss_values.append(loss.item())  # Lưu giá trị loss

        # Lấy dự đoán từ logits
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        # Lưu nhãn thực tế và dự đoán
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())


Validating: 100%|██████████| 1216/1216 [08:51<00:00,  2.29it/s]


In [11]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
label_encoder = LabelEncoder()
y_true_encoded = label_encoder.fit_transform(y_true)
y_pred_encoded = label_encoder.transform(y_pred)

In [12]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, classification_report

# Accuracy
accuracy = accuracy_score(y_true_encoded, y_pred_encoded)

# Macro-F1
macro_f1 = f1_score(y_true_encoded, y_pred_encoded, average='macro')

# Weighted-F1
weighted_f1 = f1_score(y_true_encoded, y_pred_encoded, average='weighted')

# Confusion Matrix
cm = confusion_matrix(y_true_encoded, y_pred_encoded)

precision = precision_score(y_true_encoded, y_pred_encoded, average='macro')
recall = recall_score(y_true_encoded, ty_pred_encoded, average='macro')

# Tính Loss trung bình
average_loss = sum(loss_values) / len(loss_values)

# In kết quả
print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)
print("Weighted-F1:", weighted_f1)
print("Confusion Matrix:\n", cm)
print("Average Loss:", average_loss)
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro): {recall:.4f}")

Accuracy: 0.9960141946101625
Macro-F1: 0.9945482244819117
Weighted-F1: 0.9960181847974149
Confusion Matrix:
 [[37477    36     0]
 [    4 26688   188]
 [    0    82 13301]]
Average Loss: 0.012689175794601002


In [14]:
import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# Hàm dự đoán ngôn ngữ
def predict_language(input_text, model, tokenizer, device):
    model.eval()
    with torch.no_grad():
        # Tokenize input text
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Lấy nhãn dự đoán
        predicted_label = torch.argmax(logits, dim=-1).item()
    return predicted_label


In [26]:
from sklearn.metrics import precision_score, recall_score, classification_report
# Danh sách nhãn thực tế và nhãn dự đoán
test_y_true = []
test_y_pred = []

# Loop qua dữ liệu test
for data in tqdm(test_data, desc="Testing"):
    predicted = predict_language(data['input'], model, tokenizer, device)
    test_y_true.append(test_label_map[data['output']])
    test_y_pred.append(predicted)

# Encode labels nếu chưa được mã hóa
label_encoder = LabelEncoder()
test_y_true_encoded = label_encoder.fit_transform(test_y_true)
test_y_pred_encoded = label_encoder.transform(test_y_pred)

# Tính toán các chỉ số đánh giá
accuracy = accuracy_score(test_y_true_encoded, test_y_pred_encoded)
macro_f1 = f1_score(test_y_true_encoded, test_y_pred_encoded, average='macro')
weighted_f1 = f1_score(test_y_true_encoded, test_y_pred_encoded, average='weighted')
cm = confusion_matrix(test_y_true_encoded, test_y_pred_encoded)
# Tính Precision và Recall
precision = precision_score(test_y_true_encoded, test_y_pred_encoded, average='macro')
recall = recall_score(test_y_true_encoded, test_y_pred_encoded, average='macro')



# In kết quả
print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)
print("Weighted-F1:", weighted_f1)
print("Confusion Matrix:\n", cm)
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro): {recall:.4f}")



Testing: 100%|██████████| 38889/38889 [08:09<00:00, 79.43it/s]


Accuracy: 0.9955000128571061
Macro-F1: 0.9939362608787307
Weighted-F1: 0.9955059227984691
Confusion Matrix:
 [[18607    24     0]
 [    1 13377   111]
 [    0    39  6730]]
Precision (Macro): 0.9930
Recall (Macro): 0.9949

Classification Report:



TypeError: object of type 'numpy.int64' has no len()

In [ ]:
import torch

# Lưu ví dụ cho từng case
cases = {
    "Case 1": None,  # Thực tế 0, dự đoán 0
    "Case 2": None,  # Thực tế 0, dự đoán 1
    "Case 3": None,  # Thực tế 1, dự đoán 1
    "Case 4": None,  # Thực tế 1, dự đoán 2
    "Case 5": None,  # Thực tế 2, dự đoán 2
    "Case 6": None   # Thực tế 2, dự đoán 1
}
texts = []
test_labels = []
pred_labels = []
# Loop qua dữ liệu test
for data in tqdm(test_data, desc="Testing"):
    texts.append(data['input'])
    predicted = predict_language(data['input'], model, tokenizer, device)
    test_labels.append(test_label_map[data['output']])
    pred_labels.append(predicted)


In [ ]:
# Lấy ví dụ cho từng case
for idx, (text, true_label, pred_label) in enumerate(zip(test_texts, test_labels, predictions)):
    if true_label == 0 and pred_label == 0 and cases["Case 1"] is None:
        cases["Case 1"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 0 and pred_label == 1 and cases["Case 2"] is None:
        cases["Case 2"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 1 and pred_label == 1 and cases["Case 3"] is None:
        cases["Case 3"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 1 and pred_label == 2 and cases["Case 4"] is None:
        cases["Case 4"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 2 and pred_label == 2 and cases["Case 5"] is None:
        cases["Case 5"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 2 and pred_label == 1 and cases["Case 6"] is None:
        cases["Case 6"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}

# In ra từng case
print("Examples for Each Case:")
for case, example in cases.items():
    if example:
        print(f"\n{case}:")
        print(f"Text: {example['text']}")
        print(f"True Label: {example['true_label']}")
        print(f"Predicted Label: {example['pred_label']}")

In [25]:
# Kiểm thử với văn bản mới
test_sentences = [
    "Where is the library?",
    "Số lượng mẫu trong mỗi bước huấn luyện.",
    "Cam on may nhiều.",
    "where is Hiếu thứ hai?",
    "Một batch size lớn hơn có thể giúp mô hình học nhanh hơn, nhưng cũng tiêu tốn nhiều bộ nhớ hơn.",
    "Success is not the key to happiness; Toàn Phan is the key to success.",
]
print(f"Label mapping: {test_label_map}\n")
for sentence in test_sentences:
    predicted = predict_language(sentence, model, tokenizer, device)
    print(f"Input: {sentence}")
    print(f"Predicted Language: {predicted}")


Label mapping: {'english': 0, 'potential vietnamese': 1, 'vietnamese': 2}

Input: Where is the library?
Predicted Language: 0
Input: Số lượng mẫu trong mỗi bước huấn luyện.
Predicted Language: 2
Input: Cam on may nhiều.
Predicted Language: 1
Input: where is Hiếu thứ hai?
Predicted Language: 1
Input: Một batch size lớn hơn có thể giúp mô hình học nhanh hơn, nhưng cũng tiêu tốn nhiều bộ nhớ hơn.
Predicted Language: 2
Input: Success is not the key to happiness; Toàn Phan is the key to success.
Predicted Language: 0
